<a href="https://colab.research.google.com/github/yoeda11/Ddd/blob/main/As_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install pandas

In [9]:
import pandas as pd

filepath = "/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/comments.csv"
yt = pd.read_csv(filepath)
yt.head()

,comment
0,"buat driver amd, pengalaman jujur cuma sekali ..."
1,Gpu AMD terlalu ngebut bang sampe begituan dit...
2,"Editing rendering nya besok dimasukin bang, ka..."
3,Saya pernah pake pc dengan 2 vga tersebut di a...
4,makasih banget videonya ini ngebantu buat nent...


**Preprocessing**

In [10]:
# Case Folding
# Mengubah semua teks menjadi huruf kecil.
yt["comment"] = yt["comment"].str.lower()
yt.head()

,comment
0,"buat driver amd, pengalaman jujur cuma sekali ..."
1,gpu amd terlalu ngebut bang sampe begituan dit...
2,"editing rendering nya besok dimasukin bang, ka..."
3,saya pernah pake pc dengan 2 vga tersebut di a...
4,makasih banget videonya ini ngebantu buat nent...


In [11]:
# Cleaning (Pembersihan Teks)
# Menghapus elemen yang tidak relevan

import re
def clean_text(text):
  # Handle missing / non-string
  if not isinstance(text, str):
    return ""

  # 1. Case Folding
  # text = text.lower() # sudah ada di atas

  # 2. Hapus URL
  text = re.sub(r"https?://\S+|www\.\S+", " ", text)

  # 3. Hapus mention & hashtag
  text = re.sub(r"@\w+|#\w+", " ", text)

  # 4. Hapus angka
  text = re.sub(r"\d+", " ", text)

  # 5. Hapus emoji
  text = re.sub(
      r"["
      u"\U0001F600-\U0001F64F" # emoticon
      u"\U0001F300-\U0001F5FF" # simbol & pictograph
      u"\U0001F680-\U0001F6FF" # transport
      u"\U0001F1E0-\U0001F1FF" # bendera
      "]+",
      " ",
      text
  )

  # 6. Hapus tanda baca & karakter khusus
  text = re.sub(r"[^a-z\s]", " ", text)

  # 7. Hapus karakter non-ASCII (opsional)
  text = text.encode("ascii", "ignore").decode("ascii")

  # 8. Normalisasi spasi
  text = re.sub(r"\s+", " ", text).strip()

  return text

yt["comment"] = yt["comment"].apply(clean_text)
yt.head()

,comment
0,buat driver amd pengalaman jujur cuma sekali d...
1,gpu amd terlalu ngebut bang sampe begituan dit...
2,editing rendering nya besok dimasukin bang kar...
3,saya pernah pake pc dengan vga tersebut di aca...
4,makasih banget videonya ini ngebantu buat nent...


In [12]:
# Tokenization
# Memecah kalimat menjadi kata-kata (token)
tokenized_commyt = yt["comment"].apply(lambda x: x.split())
tokenized_commyt.head(50)


,comment
0,"[buat, driver, amd, pengalaman, jujur, cuma, s..."
1,"[gpu, amd, terlalu, ngebut, bang, sampe, begit..."
2,"[editing, rendering, nya, besok, dimasukin, ba..."
3,"[saya, pernah, pake, pc, dengan, vga, tersebut..."
4,"[makasih, banget, videonya, ini, ngebantu, bua..."
5,"[udah, kapok, beli, vga, amd, udah, unit, jadi..."
6,"[saya, sudah, pke, kedua, brand, dan, mnrt, sa..."
7,"[kelebihan, nvidia, lebih, bagus, untuk, gamin..."
8,"[karena, saya, jarang, menggunakan, pc, untuk,..."
9,"[gimana, dengan, kubu, biru, bang]"


In [13]:
# Normalization (Normalisasi Kata)
# Mengubah kata tidak baku/slang menjadi baku

# Load kamus slang dari file csv
slang_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/slangindo/slang_indo.csv")
slang_dict = dict(zip(slang_df["slang"], slang_df["formal"]))

def normalized_text(tokens):
  return [slang_dict[word] if word in slang_dict else word for word in tokens]

normalized_commyt = tokenized_commyt.apply(normalized_text)
normalized_commyt.head(50)

,comment
0,"[buat, driver, amd, pengalaman, jujur, cuma, s..."
1,"[gpu, amd, terlalu, ngebut, bang, sampai, begi..."
2,"[editing, rendering, nya, besok, dimasukin, ba..."
3,"[saya, pernah, pakai , pc, dengan, vga, terseb..."
4,"[makasih, sekali , videonya, ini, ngebantu, bu..."
5,"[sudah, kapok, beli, vga, amd, sudah, unit, ja..."
6,"[saya, sudah, pakai , kedua, brand, dan, menur..."
7,"[kelebihan, nvidia, lebih, bagus, untuk, gamin..."
8,"[karena, saya, jarang, menggunakan, pc, untuk,..."
9,"[gimana, dengan, kubu, biru, bang]"


In [2]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 14.3 MB/s eta 0:00:00


In [14]:
# Stopword Removal
# Menghapus kata yang tidak punya makna penting.
# Contoh stopword:
# "dan", "di", "yang", "ke", "dari"

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

In [15]:
negation_words = {
    "tidak", "bukan", "jangan", "belum", "kurang", "tanpan"
}

stopwords = stopwords - negation_words

In [16]:
noise_words = {
    "nya", "sih", "dong", "deh", "kok", "nih", "loh", "ya", "kan", "aja", "pun", "lah"
}

stopwords = stopwords.union(noise_words)

In [17]:
def stopword_removal(tokens):
    filtered = []
    for word in tokens:
        if word not in stopwords and len(word) > 2:
            filtered.append(word)
    return filtered

In [19]:
# df['tokens_clean'] = df['tokens'].apply(stopword_removal)
stopwords_commyt = normalized_commyt.apply(stopword_removal)
stopwords_commyt.head(50)

,comment
0,"[buat, driver, amd, pengalaman, jujur, cuma, s..."
1,"[gpu, amd, terlalu, ngebut, bang, begituan, di..."
2,"[editing, rendering, besok, dimasukin, bang, t..."
3,"[pernah, pakai , vga, tersebut, acara, live, w..."
4,"[makasih, sekali , videonya, ngebantu, buat, n..."
5,"[kapok, beli, vga, amd, unit, jadi, bangkai, s..."
6,"[pakai , kedua, brand, menurut , yang , kebutu..."
7,"[kelebihan, nvidia, lebih, bagus, gaming, prof..."
8,"[jarang, menggunakan, editing, kalaupun, ngedi..."
9,"[gimana, kubu, biru, bang]"


In [ ]:
# Stemming / Lemmatization
# Mengubah kata ke bentuk dasar

In [ ]:
# Modeling (Naive Bayes / SVM / LSTM)

In [ ]:
# tokenized_commyt.to_csv("data_pandas.csv", index=False)

In [ ]:
# Normalization (Normalisasi Kata)
# Mengubah kata tidak baku/slang menjadi baku

# def load_slang_dict(path):
  # df = pd.read_csv(path)
  # df = pd.read_csv(path)

  # df["slang"] = df["slang"].str.lower()
  # df["formal"] = df["formal"].str.lower()

  # return dict(zip(df["slang"], df["formal"]))

In [ ]:
# def normalize_tokens(tokenized_commyt, slang_dict):
#   normalized = []

#   for sentence in tokenized_commyt:
#     normalized_sentence = []

#     for token in sentence:
#       if token in slang_dict:
#         normalized_sentence.append(slang_dict[token])
#       else:
#         normalized.append(token)
#     return normalized

# normal = tokenized_commyt["comment"] = yt["comment"].apply(normalize_tokens)
# normal = normalize_tokens(tokenized_commyt, load_slang_dict("/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/slangindo/slang_indo.csv"))
# tokenized_commyt.head(50)

In [ ]:
# slang_dict = load_slang_dict("/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/slangindo/slang_indo.csv")

# normal = normalize_tokens(tokenized_commyt, slang_dict)
# normal = normallized_commyt["comment"] = normal[""]
# yt["normalized"] = yt["tokens"].apply(lambda x: normalize_tokens(x, slang_dict))
# normal.head()
# print(normal)

In [ ]:
# def load_slang_dict(path):
#     slang_df = pd.read_csv(path)

    # pastikan kolom sesuai
    # slang_dict = dict(zip(slang_df['slang'], slang_df['formal']))

    # return slang_dict

# load kamus
# slang_dict = load_slang_dict('/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/slangindo/slang_indo.csv')

In [ ]:
# def normalize_tokens(tokens, slang_dict):

# def normalize_tokens(tokenized_commyt, slang_dict):

#     normalized = []

#     for word in tokenized_commyt:
#         if word in slang_dict:
#             normalized.append(slang_dict[word])
#         else:
#             normalized.append(word)

#     return normalized

In [ ]:
# df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/data_pandas.csv')

In [ ]:
# import ast

# df['tokens'] = df['tokens'].apply(ast.literal_eval)
# df['comment'] = df['comment'].apply(ast.literal_eval)
# norr.head()

In [ ]:
# nor = df['normalized'] = df['comment'].apply(lambda x: normalize_tokens(x, slang_dict))

# nor_satu = yt['normalized'] = yt['comment'].apply(lambda x: normalize_tokens(x, slang_dict))
# nor_satu.head()

In [ ]:
# nor = df['normalized'] = df['comment'].apply(lambda x: normalize_tokens(x, slang_dict))
# yt['normalized'] = yt['comment'].apply(lambda x: normalize_tokens(x, slang_dict))
# nor.head(50)